<font color=skyblue>Blur Kernel App - Display Gaussian & Defocus blur</font>

Select an image, choose kernel and parameter, then click Blur.

In [1]:
import os
import cv2
import numpy as np
import tkinter as tk
import threading
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk


def apply_gaussian_blur(img_rgb, sigma):
    """Apply Gaussian blur with adjustable sigmaX."""
    return cv2.GaussianBlur(img_rgb, (0, 0), sigmaX=float(sigma))


def disk_kernel(radius):
    """Create a normalized circular kernel for defocus blur."""
    radius = int(radius)
    size = 2 * radius + 1
    kernel = np.zeros((size, size), dtype=np.float32)
    cy, cx = radius, radius
    y, x = np.ogrid[:size, :size]
    mask = (x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2
    kernel[mask] = 1.0
    s = kernel.sum()
    return kernel / s if s > 0 else kernel


def apply_defocus_blur(img_rgb, radius):
    """Apply defocus blur using circular kernel radius."""
    return cv2.filter2D(img_rgb, -1, disk_kernel(radius))


class BlurKernelApp:
    def __init__(self, root):
        self.root = root
        self.root.title('Blur Kernel App - Gaussian & Defocus')
        self.root.geometry('1180x760')

        self.image_path = None
        self.original_np = None
        self.blurred_np = None

        self.original_tk = None
        self.blurred_tk = None

        self.kernel_type_var = tk.StringVar(value='gaussian')
        self.sigma_var = tk.DoubleVar(value=3.0)
        self.radius_var = tk.IntVar(value=5)
        self.status_var = tk.StringVar(value='Select an image, choose kernel and parameter, then click Blur.')

        # Controls row 1
        controls_1 = tk.Frame(self.root)
        controls_1.pack(fill='x', padx=12, pady=(10, 4))

        tk.Button(
            controls_1,
            text='1) Select Image',
            width=16,
            command=self.select_image,
        ).grid(row=0, column=0, padx=(0, 8), pady=4)

        self.path_label = tk.Label(controls_1, text='No image selected', anchor='w', width=95)
        self.path_label.grid(row=0, column=1, columnspan=6, sticky='w')

        # Controls row 2
        controls_2 = tk.LabelFrame(self.root, text='Kernel Type & Parameters', padx=8, pady=6)
        controls_2.pack(fill='x', padx=12, pady=(2, 4))

        tk.Radiobutton(
            controls_2,
            text='Gaussian',
            variable=self.kernel_type_var,
            value='gaussian',
            command=self._on_kernel_change,
        ).grid(row=0, column=0, padx=(0, 10), sticky='w')

        tk.Label(controls_2, text='SigmaX:').grid(row=0, column=1, sticky='e', padx=(0, 4))
        self.sigma_value_label = tk.Label(controls_2, text='3.0', width=5)
        self.sigma_value_label.grid(row=0, column=2, sticky='w', padx=(0, 4))

        self.sigma_slider = tk.Scale(
            controls_2,
            from_=0.5,
            to=20.0,
            resolution=0.5,
            orient='horizontal',
            length=260,
            variable=self.sigma_var,
            command=self._on_sigma_change,
        )
        self.sigma_slider.grid(row=0, column=3, padx=(0, 18), sticky='w')

        tk.Radiobutton(
            controls_2,
            text='Defocus (Disk)',
            variable=self.kernel_type_var,
            value='defocus',
            command=self._on_kernel_change,
        ).grid(row=0, column=4, padx=(0, 10), sticky='w')

        tk.Label(controls_2, text='Radius:').grid(row=0, column=5, sticky='e', padx=(0, 4))
        self.radius_value_label = tk.Label(controls_2, text='5', width=4)
        self.radius_value_label.grid(row=0, column=6, sticky='w', padx=(0, 4))

        self.radius_slider = tk.Scale(
            controls_2,
            from_=1,
            to=30,
            resolution=1,
            orient='horizontal',
            length=220,
            variable=self.radius_var,
            command=self._on_radius_change,
        )
        self.radius_slider.grid(row=0, column=7, sticky='w')

        # Controls row 3
        controls_3 = tk.Frame(self.root)
        controls_3.pack(fill='x', padx=12, pady=(2, 6))

        tk.Button(controls_3, text='2) Blur Image', width=16, command=self.run_blur).grid(
            row=0, column=0, padx=(0, 8)
        )

        self.save_btn = tk.Button(
            controls_3,
            text='3) Save Blurred',
            width=16,
            command=self.save_blurred,
            state='disabled',
        )
        self.save_btn.grid(row=0, column=1, padx=(0, 8))

        tk.Button(controls_3, text='Exit', width=10, command=self.root.destroy).grid(
            row=0, column=2, padx=(8, 0)
        )

        # Preview area
        preview = tk.Frame(self.root)
        preview.pack(fill='both', expand=True, padx=12, pady=4)

        left_panel = tk.Frame(preview, bd=1, relief='solid')
        right_panel = tk.Frame(preview, bd=1, relief='solid')
        left_panel.pack(side='left', fill='both', expand=True, padx=4)
        right_panel.pack(side='left', fill='both', expand=True, padx=4)

        self.left_title = tk.Label(left_panel, text='Original Image', font=('Segoe UI', 10, 'bold'))
        self.left_title.pack(pady=(8, 4))
        self.right_title = tk.Label(right_panel, text='Blurred Image', font=('Segoe UI', 10, 'bold'))
        self.right_title.pack(pady=(8, 4))

        self.left_img_lbl = tk.Label(left_panel, text='No image loaded')
        self.left_img_lbl.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        self.right_img_lbl = tk.Label(right_panel, text='No preview yet')
        self.right_img_lbl.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        tk.Label(self.root, textvariable=self.status_var, anchor='w', fg='navy').pack(
            fill='x', padx=12, pady=(0, 8)
        )

        self._on_kernel_change()

    def _on_kernel_change(self):
        is_gaussian = self.kernel_type_var.get() == 'gaussian'
        self.sigma_slider.config(state='normal' if is_gaussian else 'disabled')
        self.radius_slider.config(state='disabled' if is_gaussian else 'normal')

    def _on_sigma_change(self, _=None):
        self.sigma_value_label.config(text=f'{self.sigma_var.get():.1f}')

    def _on_radius_change(self, _=None):
        self.radius_value_label.config(text=str(int(self.radius_var.get())))

    def select_image(self):
        image_path = filedialog.askopenfilename(
            title='Select an image',
            filetypes=[('Image Files', '*.png *.jpg *.jpeg *.bmp *.tif *.tiff')],
        )
        if not image_path:
            return

        bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if bgr is None:
            messagebox.showerror('Error', f'Failed to read image:\n{image_path}')
            return

        self.image_path = image_path
        self.original_np = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        self.blurred_np = None

        self.path_label.config(text=image_path)
        self._show_in_label(self.left_img_lbl, self.original_np, 'original')

        self.right_img_lbl.config(image='', text='Click "Blur Image" to preview')
        self.right_title.config(text='Blurred Image')
        self.save_btn.config(state='disabled')
        self.status_var.set('Image loaded. Choose kernel/parameter and click Blur Image.')

    def run_blur(self):
        if self.original_np is None:
            messagebox.showwarning('Warning', 'Please select an image first.')
            return

        if self.kernel_type_var.get() == 'gaussian':
            sigma = float(self.sigma_var.get())
            self.blurred_np = apply_gaussian_blur(self.original_np, sigma)
            param_label = f'Gaussian sigmaX={sigma:.1f}'
            save_tag = f'gauss{sigma:.1f}'
        else:
            radius = int(self.radius_var.get())
            self.blurred_np = apply_defocus_blur(self.original_np, radius)
            param_label = f'Defocus radius={radius}'
            save_tag = f'defocus{radius}'

        self._show_in_label(self.right_img_lbl, self.blurred_np, 'blurred')
        self.right_title.config(text=f'Blurred ({param_label})')
        self.save_btn.config(state='normal')
        self.status_var.set(f'Blur complete using {param_label}. You can adjust and run again.')
        self._save_tag = save_tag

    def save_blurred(self):
        if self.blurred_np is None:
            messagebox.showwarning('Warning', 'No blurred image to save. Run blur first.')
            return

        base = 'image'
        if self.image_path:
            base = os.path.splitext(os.path.basename(self.image_path))[0]

        initial_name = f'{base}_blur_{getattr(self, "_save_tag", "result")}.png'

        save_path = filedialog.asksaveasfilename(
            title='Save blurred image',
            defaultextension='.png',
            initialfile=initial_name,
            filetypes=[('PNG', '*.png'), ('JPEG', '*.jpg *.jpeg'), ('Bitmap', '*.bmp')],
        )
        if not save_path:
            return

        out_u8 = np.clip(self.blurred_np, 0, 255).astype(np.uint8)
        ok = cv2.imwrite(save_path, cv2.cvtColor(out_u8, cv2.COLOR_RGB2BGR))
        if ok:
            self.status_var.set(f'Saved: {save_path}')
        else:
            messagebox.showerror('Error', f'Failed to save:\n{save_path}')

    def _show_in_label(self, label_widget, np_img, slot):
        disp = np_img
        if disp.dtype != np.uint8:
            disp = np.clip(disp, 0, 255).astype(np.uint8)

        h, w = disp.shape[:2]
        max_w, max_h = 520, 520
        scale = min(max_w / max(1, w), max_h / max(1, h), 1.0)
        nw, nh = max(1, int(w * scale)), max(1, int(h * scale))

        resized = cv2.resize(disp, (nw, nh), interpolation=cv2.INTER_AREA)
        tk_img = ImageTk.PhotoImage(Image.fromarray(resized))
        label_widget.config(image=tk_img, text='')

        if slot == 'original':
            self.original_tk = tk_img
        else:
            self.blurred_tk = tk_img


def _run_gui_thread():
    root = tk.Tk()
    BlurKernelApp(root)
    root.mainloop()


def launch_blur_gui(blocking=False):
    if blocking:
        _run_gui_thread()
        return None

    gui_thread = threading.Thread(target=_run_gui_thread, daemon=True)
    gui_thread.start()
    print('Blur kernel GUI launched in background. The cell finished immediately.')
    print('Kernel options: Gaussian (sigmaX) and Defocus (radius).')
    return gui_thread


GUI_THREAD = launch_blur_gui(blocking=False)

Blur kernel GUI launched in background. The cell finished immediately.
Kernel options: Gaussian (sigmaX) and Defocus (radius).
